# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [1]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [7]:
# Q1 (1 pt): Sentence splitting (regex + NLTK) 
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries. 
# - Then use nltk.sent_tokenize. # # Return: sentences (list of strings)

ACRONYM_DOT = "<DOT>"
# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)

def protect_acronym_dots(txt: str) -> str:
    pattern = r"\b(?:[A-Z]\.){2,}"
    def repl(m):
        letters = m.group(0).split(".")[:-1] 
        return ACRONYM_DOT.join(letters) + "."
    return re.sub(pattern, repl, txt)

def restore_acronym_dots(txt: str) -> str:
    return txt.replace(ACRONYM_DOT, ".")

# TODO: apply sent_tokenize

protected = protect_acronym_dots(text)
sentences = sent_tokenize(protected)
sentences = [restore_acronym_dots(s) for s in sentences]

print(sentences)



['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C.', 'and U.N.E.S.C.O.', 'A report valued the project at $3.2 billion.']


## Q2

In [12]:
# Q2 (1 pt): Regex normalization (pure regex + tiny helpers)
# - Acronyms: remove dots inside acronyms like U.P.C. / U.N.E.S.C.O. -> UPC / UNESCO
# - Height: X.YZm or Xm -> int(round(X.YZ*100)) centimeters
# - Money: $X.Y billion -> "x point y billion" (digits 0-9 -> words)

DIGIT_WORD = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"
}

def num_to_words_decimal(num_str: str) -> str:
    if "." in num_str:
        a, b = num_str.split(".", 1)
        a_words = DIGIT_WORD[a] if len(a) == 1 else " ".join(DIGIT_WORD[d] for d in a)
        b_words = " ".join(DIGIT_WORD[d] for d in b)
        return f"{a_words} point {b_words}"
    return DIGIT_WORD[num_str] if len(num_str) == 1 else " ".join(DIGIT_WORD[d] for d in num_str)

def normalize_text(txt: str) -> str:
    # 1) Acronyms with dots -> remove dots
    txt = re.sub(r"\b(?:[A-Z]\.){2,}", lambda m: m.group(0).replace(".", ""), txt)

    # 2) Meters -> centimeters
    def repl_meters(m):
        cm = int(round(float(m.group(1)) * 100))
        return f"{cm} centimeters"
    txt = re.sub(r"\b(\d+(?:\.\d+)?)m\b", repl_meters, txt)

    # 3) billion -> words
    def repl_billion(m):
        num_str = m.group(1)
        return f"{num_to_words_decimal(num_str)} billion"
    txt = re.sub(r"\$(\d+(?:\.\d+)?)\s*billion\b", repl_billion, txt, flags=re.IGNORECASE)

    return txt

text_norm = normalize_text(text)
print(text_norm)



In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [20]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case
def normalize_case(txt: str) -> str:
    # Step 1: Identify multiword proper nouns (capitalized words sequences)
    def repl_multiword(m):
        return m.group(0).replace(" ", "_")
    txt = re.sub(r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b", repl_multiword, txt)

    # Step 2: Lowercase everything except ALL CAPS and MixedCase
    def repl_case(m):
        token = m.group(0)
        if token.isupper() or (any(c.islower() for c in token) and any(c.isupper() for c in token)):
            return token 
        return token.lower()
    
    # Step 3: Lowercase sentence-initial capitals
    txt = re.sub(r'(?:^|(?<=\. ))([A-Z][a-z]+)', lambda m: m.group(1).lower(), txt)
    
    return txt

text_case = normalize_case(text_norm)

print (text_case)


in mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. he is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q4

In [27]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = nltk.word_tokenize(text_case)

print(tokens)


['in', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'he', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', 'A', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


## Q5

In [30]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

def remove_stopwords(tokens):
    stop_words = set(stopwords.words("english"))
    return [t for t in tokens if t.lower() not in stop_words]

tokens_nostop = remove_stopwords(tokens)

print(tokens_nostop)


['mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', '186', 'centimeters', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion', '.']


## Q6

In [ ]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = None

# print(bigrams)


## Q7

In [ ]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [ ]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
